<a href="https://colab.research.google.com/github/Meyem4/lesy_test/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets openpyxl pandas torch --quiet

In [2]:
import pandas as pd

In [3]:
df = pd.read_excel('/content/reviews_google1.xlsx')
df.head()

,query,name,google_id,place_id,location_link,reviews_link,reviews_per_score,reviews,rating,review_id,...,review_questions_Rezervace vstupenek,review_questions_Obtížnost,review_questions_Strávený čas,review_questions_Doporučené aktivity,review_questions_Typ trasy,gender,gender_female,gender_male,gender_unknown,label
0,hiking trail Českomoravské mezihoří Czech Repu...,Cinibulkova Stezka,0x47095f47c9c65249:0x1667c88bd5b67aed,ChIJSVLGyUdfCUcR7Xq21YvIZxY,https://www.google.com/maps/place/Cinibulkova+...,https://search.google.com/local/reviews?placei...,"{""1"": 3, ""2"": 3, ""3"": 23, ""4"": 145, ""5"": 761}",935.0,4.8,ChdDSUhNMG9nS0VJQ0FnSUNadVpQQnlBRRAB,...,NaN,NaN,NaN,NaN,NaN,male,0.0,1.0,0.0,neutral
1,"vyhlídka Karlovarská vrchovina Czech Republic,...",Rozhledna - vyhlídka Karla IV,0x47a09959a08b5c6b:0x491c23f975d96256,ChIJa1yLoFmZoEcRVmLZdfkjHEk,https://www.google.com/maps/place/Rozhledna+-+...,https://search.google.com/local/reviews?placei...,"{""1"": 1, ""2"": 5, ""3"": 21, ""4"": 101, ""5"": 500}",628.0,4.7,ChdDSUhNMG9nS0VJQ0FnTURnODczUWx3RRAB,...,NaN,NaN,NaN,NaN,NaN,male,0.0,1.0,0.0,positive
2,tourist attraction Jihočeské pánve Czech Repub...,Státní zámek Kratochvíle,0x4774a956b09d129b:0x273ab94b82a96d3c,ChIJmxKdsFapdEcRPG2pgku5Oic,https://www.google.com/maps/place/St%C3%A1tn%C...,https://search.google.com/local/reviews?placei...,"{""1"": 44, ""2"": 32, ""3"": 147, ""4"": 788, ""5"": 3330}",4341.0,4.7,Ci9DQUlRQUNvZENodHljRjlvT21kU2QxSnlkalZsTlVoUV...,...,NaN,NaN,NaN,NaN,NaN,male,0.0,1.0,0.0,neutral
3,viewpoint Beskydy Czech Republic,Vyhlídka Máj,0x470b83374f83bc3f:0xdf6db2c4bf584abb,ChIJP7yDTzeDC0cRu0pYv8Sybd8,https://www.google.com/maps/place/Vyhl%C3%ADdk...,https://search.google.com/local/reviews?placei...,"{""1"": 41, ""2"": 38, ""3"": 181, ""4"": 697, ""5"": 3366}",4323.0,4.7,ChZDSUhNMG9nS0VJQ0FnSUNDM0lYcUlnEAE,...,NaN,NaN,NaN,NaN,NaN,male,0.0,1.0,0.0,positive
4,"rozhledna Městské lesy Plzeň Czech Republic, M...",Rozhledna Na Chlumu,0x470af05345d3ec13:0x7c21b59cfe5a0b14,ChIJE-zTRVPwCkcRFAta_py1IXw,https://www.google.com/maps/place/Rozhledna+Na...,https://search.google.com/local/reviews?placei...,"{""1"": 21, ""2"": 27, ""3"": 87, ""4"": 200, ""5"": 398}",733.0,4.3,ChdDSUhNMG9nS0VJQ0FnSUNPdUxyT2dnRRAB,...,NaN,NaN,NaN,NaN,NaN,female,1.0,0.0,0.0,neutral


In [4]:
df['label'].value_counts()

,count
label,
positive,19312
neutral,15736
negative,9926


In [5]:
labels = sorted(df['label'].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
df['label_id'] = df['label'].map(label2id)

In [6]:
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.15, stratify=df['label_id'], random_state=42)

In [7]:
# Drop rows with missing review text
df = df.dropna(subset=['review_text'])

# Force everything to string, just in case of stray numbers/objects
df['review_text'] = df['review_text'].astype(str)

# Optional: drop rows that are empty strings or just whitespace after cleaning
df = df[df['review_text'].str.strip() != '']

df = df.reset_index(drop=True)

In [9]:
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.15, stratify=df['label'], random_state=42)

from datasets import Dataset
train_ds = Dataset.from_pandas(train_df[['review_text', 'label']].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df[['review_text', 'label']].reset_index(drop=True))

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)

Map:   0%|          | 0/27962 [00:00<?, ? examples/s]

Map:   0%|          | 0/4935 [00:00<?, ? examples/s]

In [10]:
from datasets import Dataset
from transformers import AutoTokenizer

model_name = "ufal/robeczech-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_ds = Dataset.from_pandas(train_df[['review_text', 'label_id']].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df[['review_text', 'label_id']].reset_index(drop=True))

def tokenize_fn(batch):
    return tokenizer(batch['review_text'], truncation=True, padding='max_length', max_length=256)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)

train_ds = train_ds.rename_column('label_id', 'labels')
val_ds = val_ds.rename_column('label_id', 'labels')

train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
val_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/27962 [00:00<?, ? examples/s]

Map:   0%|          | 0/4935 [00:00<?, ? examples/s]

In [11]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  504MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: ufal/robeczech-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro')
    }

In [13]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./robeczech-review-sentiment',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

In [14]:
import sys
if "torchvision" in sys.modules:
    del sys.modules["torchvision"]

In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.496290,0.399659,0.848632,0.802653
2,0.341653,0.376403,0.877609,0.839291
3,0.248253,0.364911,0.896859,0.864416
4,0.188306,0.388963,0.905572,0.875868


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6992, training_loss=0.34156986232207626, metrics={'train_runtime': 5337.8317, 'train_samples_per_second': 20.954, 'train_steps_per_second': 1.31, 'total_flos': 1.4714354773020672e+16, 'train_loss': 0.34156986232207626, 'epoch': 4.0})

In [16]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.188306,0.388963,4,0.905572,0.875868


{'eval_loss': 0.38896334171295166,
 'eval_accuracy': 0.9055724417426545,
 'eval_f1_macro': 0.8758677944223593}

In [17]:
preds_output = trainer.predict(val_ds)
preds = np.argmax(preds_output.predictions, axis=-1)
true_labels = preds_output.label_ids

from sklearn.metrics import classification_report
print(classification_report(true_labels, preds, target_names=id2label.values()))

              precision    recall  f1-score   support

    negative       0.88      0.84      0.86       921
     neutral       0.83      0.79      0.81      1117
    positive       0.94      0.97      0.95      2897

    accuracy                           0.91      4935
   macro avg       0.88      0.87      0.88      4935
weighted avg       0.90      0.91      0.90      4935



In [18]:
save_path = '/content/robeczech-review-sentiment-finetuned'
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/robeczech-review-sentiment-finetuned/tokenizer_config.json',
 '/content/robeczech-review-sentiment-finetuned/tokenizer.json')

In [28]:
import os
print(os.listdir('/content'))

['.config', 'robeczech-review-sentiment', 'robeczech-review-sentiment-finetuned', 'reviews_google1.xlsx', 'sample_data']


In [32]:
from transformers import pipeline

classifier = pipeline('text-classification', model=save_path, tokenizer=save_path, device=0)



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [33]:
df_all = pd.read_excel('/content/reviews_google1.xlsx')  # match exact filename from the listdir output

In [37]:
from transformers import pipeline

classifier = pipeline('text-classification', model=save_path, tokenizer=save_path, device=0)

df_all = pd.read_excel('/content/reviews_google1.xlsx')

# Handle missing review text before running inference
df_all = df_all.dropna(subset=['review_text'])
df_all['review_text'] = df_all['review_text'].astype(str)
df_all = df_all[df_all['review_text'].str.strip() != '']
df_all = df_all.reset_index(drop=True)

results = classifier(df_all['review_text'].tolist(), truncation=True, max_length=256, batch_size=16)

df_all['predicted_sentiment'] = [r['label'] for r in results]
df_all['confidence'] = [r['score'] for r in results]

df_all.to_excel('/content/reviews_google1_scored.xlsx', index=False)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]